# Data quality — review before you run the pipeline

Reads every raw questionnaire, the same files notebook 1 reads, and finds the
two things worth a human's judgement before a full run:

* **labels the dictionary has never seen** — a column header or a value with
  no good match at any confidence, in either language;
* **Value cells that are not a number and that `clean_values()` could not
  make sense of on its own** — everything it *can* resolve (a unit phrase, a
  placeholder, a sum) happens automatically in the real run and needs no
  review; this is only the residue.

Nothing is changed here. Running this notebook writes one file,
**`data_quality_review.txt`**, with a suggested correction against every
finding that you can accept, edit, or skip:

1. Run this notebook. It writes `data_quality_review.txt` beside the codes
   folder.
2. Open it in any text editor. Each finding has one `CORRECTION:` line —
   leave it to accept the suggestion, type over it to supply your own, or
   replace it with `SKIP` to leave that one for later.
3. Save the file.
4. Tell Claude: **apply data_quality_review.txt**. Claude re-reads that exact
   file — nothing else — and writes your decisions into `translation
   dict.xlsx` (labels) and `value corrections.xlsx` (values). Both are backed
   up first.
5. Run the real pipeline. Every label and value you confirmed is now an exact
   match, so notebook 1 uses it directly instead of guessing.

A third section, **structural problems**, is reporting-only — a duplicate
column header or a stray space in a merge key breaks parsing itself and has
to be fixed in the source Excel file by hand, so there is no correction to
apply.

**This replaces `Compendium_Data_Quality.ipynb`'s old raw-questionnaire
checks.** Nothing here writes to `pipeline_inconsistencies.txt` — that file
is the pipeline notebooks' own record of *what they did on their last real
run*; this is a review of the input, produced before any of them run, and
kept in its own file so opening one never leaves you looking at a stale mix
of the two.


In [ ]:
"""
CELL: Imports and logging setup.
"""
import difflib
import logging
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("compendium")


## Config

In [ ]:
"""
CELL: Configuration - the same paths and constants notebook 1 uses, since this
notebook has to see exactly what notebook 1 would see.
"""
DATA_COLLECTOR_PATH = Path.home() / "OneDrive - United Nations" / "Desktop" / "DSS" / "DATA COLLECTOR"
TRANSLATION_DICT_PATH = DATA_COLLECTOR_PATH / "translation dict.xlsx"
VALUE_CORRECTIONS_PATH = DATA_COLLECTOR_PATH / "value corrections.xlsx"
COMPENDIUM_PATH = Path.home() / "OneDrive - United Nations" / "Desktop" / "DSS" / "COMPENDIUM-ARAB SOCIETY"

QUESTIONNAIRE_PREFIX = "datacollector_received_quest_"
LANGUAGES = ["AR", "EN"]
DEFAULT_LANGUAGE = "AR"

# Leave as None to check every chapter found, or restrict e.g. ["Population"].
CHAPTERS = None

# A fuzzy match must score at least this well (0 to 1) to be used automatically
# by the real pipeline. Below it is exactly what this notebook reviews - same
# number as notebook 1's, because a label just above the line needs no review
# and one just below it does.
FUZZY_MATCH_CUTOFF = 0.6

# Column names the pipeline creates itself, written here in Arabic.
YEAR_COLUMN = "السنة"
VALUE_COLUMN = "العدد"
CHAPTER_COLUMN = "الفصل"

MERGE_COLUMNS = ["السنة", "المؤشر", "الدولة"]

# Never fuzzy-matched by the real pipeline either: two citations differing by
# one digit score high enough to overwrite each other. A Source value still
# gets reviewed here if the dictionary has no *exact* translation for it, but
# never with a fuzzy-matched suggestion pre-filled.
COLUMNS_NOT_FUZZY_MATCHED = ["المصدر"]

# Where this notebook writes its one output.
REVIEW_PATH = COMPENDIUM_PATH / "data_quality_review.txt"


## The dictionary

In [ ]:
"""
CELL: Load translation dict.xlsx - identical to notebook 1's own loader, since
a label reviewed here has to be checked against the exact same vocabulary
notebook 1 will use.
"""


def load_dictionary():
    dict_df = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")

    column_map, value_map = {}, {}
    for arabic_column in dict_df["col_ar"].dropna().unique():
        rows = dict_df[dict_df["col_ar"] == arabic_column]
        column_map[arabic_column] = rows["col_en"].iloc[0]
        value_map[arabic_column] = {
            arabic: english
            for arabic, english in zip(rows["val_ar"], rows["val_en"])
            if pd.notna(arabic)
        }

    english_columns = {}
    english_values = {}
    for english_column in dict_df["col_en"].dropna().unique():
        rows = dict_df[dict_df["col_en"] == english_column]
        english_columns[english_column] = english_column
        english_values[english_column] = {
            str(v): str(v) for v in rows["val_en"].dropna().unique()
        }

    chapter_rows = dict_df[dict_df["col_en"] == "Chapter"]
    chapter_to_arabic = dict(zip(chapter_rows["val_en"], chapter_rows["val_ar"]))

    return (column_map, value_map), (english_columns, english_values), chapter_to_arabic


DICTIONARY_AR_TO_EN, ENGLISH_VOCABULARY, CHAPTER_TO_ARABIC = load_dictionary()


def vocabulary(language):
    """The known column names and values for a language, as
    (column_names, values_by_column) - what a raw label is checked against."""
    return DICTIONARY_AR_TO_EN if language == "AR" else ENGLISH_VOCABULARY


def column_name_for(english_name, language):
    """Any dictionary column, spelled for the given language."""
    if language == "EN":
        return english_name
    english_to_arabic = {en: ar for ar, en in DICTIONARY_AR_TO_EN[0].items()}
    return english_to_arabic.get(english_name, english_name)


def column_name(arabic_name, language):
    """One of the pipeline's own column names, spelled for the given language."""
    arabic_to_english, _ = DICTIONARY_AR_TO_EN
    return arabic_name if language == "AR" else arabic_to_english[arabic_name]


def known_english_columns():
    """Every English column name the dictionary knows - the vocabulary a
    column-name finding's correction has to be one of."""
    arabic_to_english, _ = DICTIONARY_AR_TO_EN
    return sorted(set(arabic_to_english.values()))


arabic_columns, _ = DICTIONARY_AR_TO_EN
logger.info(f"Dictionary loaded: {len(arabic_columns)} column names, Arabic -> English")


## Recognizing a Value cell notebook 1 could not make sense of

In [ ]:
"""
CELL: clean_one_value() - identical to notebook 1's, so "not a number" means
the same thing here as it will on the real run.

Only clean_one_value() itself is needed, not the table-level clean_values():
this notebook checks one raw cell at a time as it reads each sheet, rather
than a whole reshaped column at once.
"""

PLACEHOLDERS = {"-", "--", "---", "..", "...", "n/a", "na", "n.a.", "nil", "none"}

NUMBER_IN_TEXT = re.compile(r"[-+]?\d[\d\s, ]*(?:\.\d+)?")

SUM_EXPRESSION = re.compile(r"^\d[\d\s,]*(?:\.\d+)?(?:\s*\+\s*\d[\d\s,]*(?:\.\d+)?)+$")

UNIT_MULTIPLIERS = [
    (re.compile(r"بالمليون|بالملايين"), 1_000_000, "in millions"),
    (re.compile(r"بالال?ف"), 1_000, "in thousands"),
]


def load_value_corrections():
    """Raw Value text a person has already confirmed a reading for, from a
    previous run of this same notebook's review-and-apply loop. A cell that
    matches one is not a new finding - it is already answered, and
    clean_one_value() below returns that answer instead of flagging it again.
    """
    if not VALUE_CORRECTIONS_PATH.exists():
        return {}
    table = pd.read_excel(VALUE_CORRECTIONS_PATH, engine="openpyxl")
    corrections = {}
    for chapter, raw, corrected in zip(
        table["chapter"], table["raw_value"], table["corrected_value"]
    ):
        if pd.isna(chapter) or pd.isna(raw) or pd.isna(corrected):
            continue
        corrections[(str(chapter).strip(), str(raw).strip())] = str(corrected).strip()
    return corrections


VALUE_CORRECTIONS = load_value_corrections()
if VALUE_CORRECTIONS:
    logger.info(f"Value corrections already on file: {len(VALUE_CORRECTIONS):,} "
                f"(those cells are already answered and will not be reviewed again)")


def unit_multiplier(text):
    for pattern, factor, meaning in UNIT_MULTIPLIERS:
        if pattern.search(text):
            return factor, meaning
    return 1, None


def scaled_text(number, factor):
    value = float(number) * factor
    return str(int(value)) if value.is_integer() else str(round(value, 10))


def clean_one_value(raw, chapter=None):
    """Return (cleaned, note, factor) - see notebook 1's cell of the same name
    for what each means. This copy exists so this notebook needs nothing from
    notebook 1 at import time; keep the two in step by hand if either changes.
    """
    if pd.isna(raw):
        return raw, None, 1

    text = str(raw).replace("\xa0", " ").strip()
    if text == "":
        if str(raw) == "":
            return raw, None, 1
        return "", f"{str(raw)!r} holds only whitespace - means no data, blanked", 1

    plain = re.sub(r"[\s,]", "", text)
    try:
        float(plain)
        return raw, None, 1
    except ValueError:
        pass

    confirmed = VALUE_CORRECTIONS.get((chapter, text))
    if confirmed is not None:
        return confirmed, f"{text!r} -> {confirmed} (confirmed reading, from value corrections.xlsx)", 1

    if text.lower() in PLACEHOLDERS:
        return "", f"{text!r} means no data - blanked", 1

    if SUM_EXPRESSION.match(text):
        parts = [float(re.sub(r"[\s,]", "", part)) for part in text.split("+")]
        total = scaled_text(sum(parts), 1)
        return total, (f"{text!r} -> {total} (read as a sum: "
                       f"{' + '.join(f'{part:g}' for part in parts)})"), 1

    match = NUMBER_IN_TEXT.search(text)
    if match:
        number = re.sub(r"[\s, ]", "", match.group(0))
        try:
            float(number)
        except ValueError:
            return raw, f"{text!r} is not a number and was left as it is", 1
        removed = text.replace(match.group(0), "").strip()

        factor, meaning = unit_multiplier(removed) if removed else (1, None)
        if factor != 1:
            value = scaled_text(number, factor)
            return value, (f"{text!r} -> {value} (dropped {removed!r}, meaning {meaning} "
                           f"- multiplied by {factor:,})"), factor

        return number, (f"{text!r} -> {number} (dropped {removed!r})"
                        if removed else f"{text!r} -> {number}"), 1

    return raw, f"{text!r} is not a number and was left as it is", 1


def needs_review(raw, chapter=None):
    """True only for the residue clean_one_value() could not make sense of -
    everything else is either already fine or already handled automatically
    by the real pipeline, and reviewing it here would just be noise.
    """
    cleaned, note, _ = clean_one_value(raw, chapter)
    return note is not None and str(cleaned) == str(raw)


## The three checks

In [ ]:
"""
CELL: check_labels(), check_values(), check_structure() - the three checks,
run once per sheet as it is read. Nothing here corrects anything; each only
records what it finds.
"""

# Raw hits, one per unique value per sheet - the same granularity
# correct_with_dictionary() itself works at. Aggregated across the whole run
# into one finding per (chapter, column, raw text) before the review file is
# written, so a label repeated across fifty sheets appears once.
LABEL_HITS = []
VALUE_HITS = []
STRUCTURE_PROBLEMS = []


def best_match(text, choices):
    best_choice, best_score = None, -1
    for choice in choices:
        score = difflib.SequenceMatcher(None, str(text), str(choice)).ratio()
        if score > best_score:
            best_choice, best_score = choice, score
    return best_choice, best_score


def check_labels(table, language, chapter, file_name, sheet_name):
    """Column names and values with no good match in the dictionary - the
    same two passes as notebook 1's correct_with_dictionary(), except nothing
    is renamed and only the below-cutoff cases are kept. A label notebook 1
    would fix on its own needs no review; this is only what it would give up
    on and leave as it found it.
    """
    known_columns, known_values_by_column = vocabulary(language)
    never_guessed = [column_name(c, language) for c in COLUMNS_NOT_FUZZY_MATCHED]

    for column in table.columns:
        if column in known_columns:
            continue

        match, score = best_match(column, known_columns.keys())
        if match is None or score < FUZZY_MATCH_CUTOFF:
            LABEL_HITS.append({
                "scope": "column", "chapter": chapter, "col_ar": None,
                "raw": column, "language": language,
                "closest": match, "closest_en": None, "score": score,
                "file": file_name, "sheet": sheet_name,
            })

    for column in table.columns:
        if column in never_guessed:
            # Source is only ever a translation concern on the ARABIC side -
            # an English questionnaire's citation is already the target
            # language and needs nothing. Even there, plenty of Arabic
            # questionnaires cite an English source ("MICS 2022", a URL) -
            # already correct as it stands, exactly like notebook 3's own
            # find_untranslated() treats it. What is left after both of those
            # is flagged, but never with a fuzzy-matched suggestion: two
            # citations differing by one digit would otherwise score high
            # enough to suggest one for the other.
            if language == "AR":
                known_values = known_values_by_column.get(column) or {}
                for value in table[column].dropna().unique():
                    if value in known_values or not looks_arabic(value):
                        continue
                    LABEL_HITS.append({
                        "scope": "source", "chapter": chapter, "col_ar": column,
                        "raw": value, "language": language,
                        "closest": None, "closest_en": None, "score": None,
                        "file": file_name, "sheet": sheet_name,
                    })
            continue

        known_values = known_values_by_column.get(column)
        if not known_values:
            continue  # not a dictionary column, or has no fixed vocabulary (e.g. Year, Value)

        for value in table[column].dropna().unique():
            if value in known_values:
                continue

            match, score = best_match(value, known_values.keys())
            if match is None or score < FUZZY_MATCH_CUTOFF:
                LABEL_HITS.append({
                    "scope": "value", "chapter": chapter, "col_ar": column,
                    "raw": value, "language": language,
                    "closest": match, "closest_en": known_values.get(match) if match else None,
                    "score": score, "file": file_name, "sheet": sheet_name,
                })


def check_values(table, language, chapter, file_name, sheet_name):
    """Value cells that are not a number and that clean_one_value() could not
    make sense of on its own - see needs_review() above for exactly which
    cases that is.
    """
    value_column = column_name(VALUE_COLUMN, language)
    if value_column not in table.columns:
        return

    indicator_column = column_name_for("Indicator", language)
    country_column = column_name_for("Country", language)
    year_column = column_name(YEAR_COLUMN, language)

    for position, raw in table[value_column].items():
        if pd.isna(raw) or not needs_review(raw, chapter):
            continue
        where = {}
        for field, column in [("country", country_column),
                              ("indicator", indicator_column),
                              ("year", year_column)]:
            if column in table.columns:
                where[field] = table.at[position, column]
        VALUE_HITS.append({
            "chapter": chapter, "raw": str(raw).replace("\xa0", " ").strip(),
            "file": file_name, "sheet": sheet_name, **where,
        })


# Cover / metadata tabs have no "index" header row and are skipped by design
# - not a problem, just not a data sheet. Matched by name because there is
# nothing else to go on: a tab with no header row looks structurally
# identical whether it is an expected cover page or a genuinely broken data
# sheet, and only the name tells the two apart (Iraq's and Jordan's legacy
# Health sheets have no "index" row either, and ARE a real problem - see
# Known issues in CLAUDE.md). The metadata tab's name varies by file
# (a leading "_", a trailing " SDG", "_sdg" vs "_SDG", "المفصلة" vs
# "المجمعة") but always contains this core phrase, so it is matched as a
# substring rather than a fixed set of exact names.
COVER_TAB_NAMES = {"العنوان", "قائمة الجداول", "كيفية الإستخدام"}
COVER_TAB_MARKER = "البيانات الوصفية"


def is_cover_tab(sheet_name):
    name = str(sheet_name).strip()
    return name in COVER_TAB_NAMES or COVER_TAB_MARKER in name


def check_structure_error(chapter, language, file_name, sheet_name, error):
    """A sheet (or file) that could not be read at all."""
    STRUCTURE_PROBLEMS.append({
        "chapter": chapter, "language": language, "file": file_name,
        "sheet": sheet_name, "problem": "cannot read the sheet",
        "detail": f"{type(error).__name__}: {error}",
    })


def check_structure_raw(raw_sheet, header_rows, chapter, language, file_name, sheet_name):
    """Problems that do not raise an exception but corrupt the sheet anyway -
    ported from the old Compendium_Data_Quality.ipynb's
    check_questionnaires(). A duplicate column header does not crash
    extract_tables(); it silently produces a stray extra column instead, which
    is exactly why this cannot wait to be caught as an error. Reporting only:
    every one of these has to be fixed in the source Excel file, not here.
    """
    merge_keys = {"السنة", "المؤشر", "الدولة", "Year", "Indicator", "Country"}
    here = {"chapter": chapter, "language": language, "file": file_name, "sheet": sheet_name}

    names = raw_sheet.iloc[header_rows[0]].dropna().astype(str).str.strip().tolist()

    duplicated = {n for n in names if names.count(n) > 1}
    if duplicated:
        STRUCTURE_PROBLEMS.append({**here, "problem": "duplicate column header",
                                   "detail": f"{sorted(duplicated)} - the second becomes a stray column"})

    year_like = [n for n in names if re.fullmatch(r"\d{4}(\.\d+)?", n) and not n.isdigit()]
    if year_like:
        STRUCTURE_PROBLEMS.append({**here, "problem": "year column is not a plain number",
                                   "detail": f"{year_like} - will be treated as a label, not unpivoted"})

    if not any(n.isdigit() for n in names):
        STRUCTURE_PROBLEMS.append({**here, "problem": "no year columns", "detail": "nothing to unpivot"})

    positions = raw_sheet.iloc[header_rows[0]]
    for position, name in positions.items():
        if not isinstance(name, str) or name.strip() not in merge_keys:
            continue
        if name != name.strip():
            STRUCTURE_PROBLEMS.append({**here, "problem": "whitespace in a merge-key header",
                                       "detail": f"{name!r} - breaks the source merge"})
        body = raw_sheet.index[raw_sheet[0] == "1"]
        values = raw_sheet.loc[body, position].dropna()
        untidy = [v for v in values.unique() if isinstance(v, str) and v != v.strip()]
        if untidy:
            STRUCTURE_PROBLEMS.append({**here, "problem": "whitespace in merge-key values",
                                       "detail": f"{len(untidy)} value(s), e.g. {untidy[0]!r}"})


## Reading a raw sheet

In [ ]:
"""
CELL: Reading a raw sheet - identical to notebook 1's extract/detect/reshape
cells, so this notebook checks exactly the table notebook 1 would build,
before any correction is applied to it. No file is written from here.
"""


def questionnaire_folders():
    found = []
    for folder in sorted(DATA_COLLECTOR_PATH.glob(f"{QUESTIONNAIRE_PREFIX}*")):
        if not folder.is_dir():
            continue
        language = folder.name[len(QUESTIONNAIRE_PREFIX):].strip().upper()
        if language not in LANGUAGES:
            logger.warning(f"Skipping {folder.name}: '{language}' is not one of {LANGUAGES}")
            continue
        found.append((language, folder))
    return found


def discover_chapters():
    names = set()
    for _, questionnaire_root in questionnaire_folders():
        for child in questionnaire_root.iterdir():
            if child.is_dir() and any(
                f for f in child.glob("*.xlsx") if not f.name.startswith("~$")
            ):
                names.add(child.name)
    return sorted(names)


def chapters_to_process():
    if CHAPTERS:
        return list(CHAPTERS)
    found = discover_chapters()
    logger.info(f"Chapters discovered on disk: {found}")
    return found


def looks_arabic(text):
    """True if the text contains at least one Arabic letter. U+0600-U+06FF is
    the Arabic Unicode block; English text has nothing in it. Identical to
    notebook 1's own function of the same name."""
    return any("؀" <= character <= "ۿ" for character in str(text))


def detect_language(table):
    names = [str(c) for c in table.columns if not str(c).isdigit()]
    if not names:
        return DEFAULT_LANGUAGE
    arabic_names = sum(1 for name in names if looks_arabic(name))
    return "AR" if arabic_names > len(names) / 2 else "EN"


def extract_tables(raw_sheet):
    header_rows = raw_sheet.index[raw_sheet[0] == "index"].tolist()
    data_header_row, source_header_row = header_rows[0], header_rows[1]

    data_columns = raw_sheet.iloc[data_header_row].dropna().str.strip()
    data_table = raw_sheet[raw_sheet[0] == "1"][data_columns.index].copy()
    data_table.columns = data_columns.values
    data_table = data_table.drop(columns=["index"])

    source_columns = raw_sheet.iloc[source_header_row].dropna().str.strip()
    source_table = raw_sheet[raw_sheet[0] == "2"][source_columns.index].copy()
    source_table.columns = source_columns.values
    source_table = source_table.drop(columns=["index"])

    return data_table, source_table


def reshape_and_merge(data_table, source_table, language):
    id_columns = [c for c in data_table.columns if not str(c).isdigit()]
    year_columns = [c for c in data_table.columns if str(c).isdigit()]

    long_table = data_table.melt(
        id_vars=id_columns,
        value_vars=year_columns,
        var_name=column_name(YEAR_COLUMN, language),
        value_name=column_name(VALUE_COLUMN, language),
    )

    merge_columns = [column_name(c, language) for c in MERGE_COLUMNS]
    merge_columns = [c for c in merge_columns if c in long_table.columns and c in source_table.columns]

    long_table = long_table.copy()
    source_table = source_table.copy()
    for column in merge_columns:
        long_table[column] = long_table[column].astype(str).str.replace("\xa0", " ").str.strip()
        source_table[column] = source_table[column].astype(str).str.replace("\xa0", " ").str.strip()

    return pd.merge(long_table, source_table, on=merge_columns, how="left")


def raw_sheets(chapter, language, questionnaire_root):
    """Every (file_name, sheet_name, table) notebook 1 would build for one
    chapter folder, in the order it would build them.

    A sheet that cannot be read at all, or that reads but is structurally
    broken (duplicate header, stray whitespace, ...), is reported through
    check_structure_error() / check_structure_raw() and skipped - nothing is
    yielded for it, so callers only ever see a usable table.
    """
    folder = questionnaire_root / chapter
    if not folder.exists():
        return

    for file_path in sorted(f for f in folder.glob("*.xlsx") if not f.name.startswith("~$")):
        try:
            xls = pd.ExcelFile(file_path, engine="openpyxl")
        except Exception as error:
            check_structure_error(chapter, language, file_path.name, "-", error)
            continue

        for sheet_name in xls.sheet_names:
            try:
                raw_sheet = pd.read_excel(xls, sheet_name=sheet_name, header=None, dtype=str)
            except Exception as error:
                check_structure_error(chapter, language, file_path.name, sheet_name, error)
                continue

            header_rows = raw_sheet.index[raw_sheet[0] == "index"].tolist()
            if len(header_rows) < 2:
                if not is_cover_tab(sheet_name):
                    # No "index" row and not a recognized cover-tab name: this
                    # is what notebook 1 itself would hit as an IndexError at
                    # its "extract" step - a real problem, not an expected skip.
                    STRUCTURE_PROBLEMS.append({
                        "chapter": chapter, "language": language,
                        "file": file_path.name, "sheet": sheet_name,
                        "problem": "no index header row, and not a recognized cover tab",
                        "detail": f"only {len(header_rows)} 'index' row(s) found "
                                  f"(need 2) - likely a legacy sheet layout, needs "
                                  f"a look in the source file",
                    })
                continue

            check_structure_raw(raw_sheet, header_rows, chapter, language,
                                file_path.name, sheet_name)
            try:
                data_table, source_table = extract_tables(raw_sheet)
                table = reshape_and_merge(data_table, source_table, language)
            except Exception as error:
                check_structure_error(chapter, language, file_path.name, sheet_name, error)
                continue

            yield file_path.name, sheet_name, table


## run_checks()

In [ ]:
"""
CELL: run_checks() - read every questionnaire and collect every finding.
"""


def run_checks():
    LABEL_HITS.clear()
    VALUE_HITS.clear()
    STRUCTURE_PROBLEMS.clear()

    folders = questionnaire_folders()
    chapters = chapters_to_process()
    print(f"Questionnaire folders found: {[f'{lang} ({f.name})' for lang, f in folders]}")

    sheets_read = 0
    total_steps = len(folders) * len(chapters)
    step_number = 0
    for language, questionnaire_root in folders:
        print(f"\n=== {language}  ({questionnaire_root.name}) ===")
        for chapter in chapters:
            step_number += 1
            bar = "#" * step_number + "-" * (total_steps - step_number)
            print(f"[{bar}] {step_number}/{total_steps}  {language}/{chapter}")
            for file_name, sheet_name, table in raw_sheets(chapter, language, questionnaire_root):
                check_labels(table, language, chapter, file_name, sheet_name)
                check_values(table, language, chapter, file_name, sheet_name)
                sheets_read += 1

    logger.info(f"Read {sheets_read:,} data sheet(s)")
    return sheets_read


def aggregate_labels():
    """LABEL_HITS collapsed to one finding per (scope, column, raw text) -
    the same typo showing up in fifty sheets is one thing to teach the
    dictionary, not fifty.
    """
    grouped = {}
    order = []
    for hit in LABEL_HITS:
        key = (hit["scope"], hit["col_ar"], hit["raw"])
        if key not in grouped:
            grouped[key] = {**hit, "locations": []}
            order.append(key)
        grouped[key]["locations"].append((hit["chapter"], hit["file"], hit["sheet"]))
    return [grouped[key] for key in order]


def aggregate_values():
    """VALUE_HITS collapsed to one finding per (chapter, raw text) - scoped to
    the chapter, unlike labels, because value corrections.xlsx itself is.
    """
    grouped = {}
    order = []
    for hit in VALUE_HITS:
        key = (hit["chapter"], hit["raw"])
        if key not in grouped:
            grouped[key] = {**hit, "locations": [], "examples": []}
            order.append(key)
        grouped[key]["locations"].append((hit["file"], hit["sheet"]))
        example = " ".join(f"{field}={hit[field]}" for field in ("country", "indicator", "year")
                           if field in hit and pd.notna(hit[field]))
        if example and example not in grouped[key]["examples"]:
            grouped[key]["examples"].append(example)
    return [grouped[key] for key in order]


## write_review()

In [ ]:
"""
CELL: write_review() - the one file this notebook produces.

Every finding gets one CORRECTION line with a suggestion pre-filled where
there is one. The bracketed [L0001 ...] / [V0001 ...] line on top of each
finding is read back by apply_review() - it is safe to leave alone and unsafe
to retype, because a retyped Arabic string that differs by one invisible
character (a non-breaking space, a different letter form) silently matches
nothing. Only the CORRECTION line is meant to be edited.
"""

SKIP = "SKIP"


def format_locations(locations, limit=3):
    """A short, readable sample of where a finding was seen.

    Each location is (file, sheet) for a value finding, or (chapter, file,
    sheet) for a label finding - only the last two elements are shown, since
    a label's chapter is already named on its [L####] line.
    """
    shown = list(dict.fromkeys(locations))  # de-duplicate, keep order
    text = ", ".join(f"{loc[-2]} ({loc[-1]})" for loc in shown[:limit])
    if len(shown) > limit:
        text += f", and {len(shown) - limit} more"
    return text


def write_review(labels, values, structure, path=None):
    path = path or REVIEW_PATH
    stamp = pd.Timestamp.now().strftime("%d %B %Y, %H:%M")
    lines = []

    lines += [
        "DATA QUALITY REVIEW", "=" * 78,
        f"generated {stamp}, from the raw questionnaires in",
        "datacollector_received_quest_AR and _EN. Nothing has been changed - this",
        "is a review, not a run of the pipeline.",
        "",
        "HOW TO USE THIS FILE",
        "  Every finding below has one CORRECTION line. To act on a finding:",
        "    - leave the CORRECTION line exactly as printed, to accept the suggestion",
        "    - type over it with your own answer instead",
        f"    - replace it with {SKIP} to leave that one for later",
        "  Save the file, then tell Claude: apply data_quality_review.txt",
        "  Claude re-reads this exact file, and only this file - nothing is written",
        "  to the dictionary or anywhere else until you do that.",
        "",
        "  Leave the bracketed [L0001 ...] / [V0001 ...] line on each finding alone.",
        "  It is how Claude finds its way back to this exact finding; retyping the",
        "  Arabic on a RAW line risks an invisible difference (a non-breaking space,",
        "  a different letter form) that would silently fail to match anything.",
        "",
        "  A label finding teaches translation dict.xlsx one new row, and every",
        "  sheet using that label is fixed from the next pipeline run on. A value",
        "  finding teaches value corrections.xlsx one exact raw cell text; the same",
        "  text anywhere else in this chapter is fixed the same way, automatically,",
        "  from the next run on.",
        "",
        "  Known English column names, for a 'column' finding's correction:",
        "    " + ", ".join(known_english_columns()),
        "",
    ]

    # --------------------------------------------------------- section 1
    lines += ["#" * 80, f"1. LABELS NOT FOUND IN THE DICTIONARY, EXACTLY  ({len(labels)})", "#" * 80]
    if not labels:
        lines += ["", "  Nothing found - every label in these files matched exactly."]
    else:
        lines += [
            "A column name or value with no exact entry in translation dict.xlsx.",
            "CLOSEST is the nearest dictionary entry difflib could find, whatever its",
            "score - shown as a starting point even when low, not a confident guess.",
            "A 'source' finding never gets a CLOSEST: two citations differing by one",
            "digit score high enough to suggest one for the other, so the real",
            "pipeline never fuzzy-matches Source either.",
        ]
        for i, finding in enumerate(labels, start=1):
            scope = finding["scope"]
            where = format_locations(finding["locations"])
            lines.append("")
            if scope == "column":
                lines.append(f"[L{i:04d} scope=column chapter={finding['chapter']}]")
                lines.append(f"  a raw column header not in the dictionary")
                lines.append(f"  seen in:     {where}")
                lines.append(f"  RAW:         {finding['raw']}")
                if finding["closest"] is not None:
                    lines.append(f"  CLOSEST:     {finding['closest']} (score {finding['score']:.2f})")
                lines.append(f"  CORRECTION (English column name this maps to): {finding['closest_en'] or ''}")
            elif scope == "source":
                lines.append(f"[L{i:04d} scope=source chapter={finding['chapter']} col_ar={finding['col_ar']}]")
                lines.append(f"  Source citation with no exact translation yet")
                lines.append(f"  seen in:     {where}")
                lines.append(f"  RAW:         {finding['raw']}")
                lines.append(f"  CORRECTION (type the English citation): ")
            else:
                lines.append(f"[L{i:04d} scope=value chapter={finding['chapter']} col_ar={finding['col_ar']}]")
                lines.append(f"  seen in:     {where}")
                lines.append(f"  RAW:         {finding['raw']}")
                if finding["closest"] is not None:
                    lines.append(f"  CLOSEST:     {finding['closest']} -> {finding['closest_en']} "
                                 f"(score {finding['score']:.2f})")
                lines.append(f"  CORRECTION (English translation): {finding['closest_en'] or ''}")

    # --------------------------------------------------------- section 2
    lines += ["", "#" * 80, f"2. VALUES THAT ARE NOT PLAIN NUMBERS  ({len(values)})", "#" * 80]
    if not values:
        lines += ["", "  Nothing found - clean_one_value() made sense of every Value cell on its own."]
    else:
        lines += [
            "Everything clean_values() can already resolve on its own - a unit phrase,",
            "a placeholder, a sum - happens automatically on the real run and needs no",
            "review. This is only the residue it could not make sense of by itself.",
            "Scoped to one chapter: the same raw text in a different chapter is",
            "reviewed, and answered, separately.",
        ]
        for i, finding in enumerate(values, start=1):
            where = format_locations(finding["locations"])
            example = finding["examples"][0] if finding["examples"] else ""
            lines.append("")
            lines.append(f"[V{i:04d} chapter={finding['chapter']}]")
            lines.append(f"  seen in:     {where}" + (f", e.g. {example}" if example else ""))
            lines.append(f"  RAW:         {finding['raw']}")
            lines.append(f"  CORRECTION (the number this cell should read):  ")

    # --------------------------------------------------------- section 3
    lines += ["", "#" * 80, f"3. STRUCTURAL PROBLEMS IN THE RAW FILES  ({len(structure)})", "#" * 80]
    if not structure:
        lines += ["", "  Nothing found."]
    else:
        lines += [
            "Reporting only - these break parsing itself and have to be fixed in the",
            "source Excel file by hand. There is nothing here to type a correction into.",
        ]
        by_problem = defaultdict(list)
        for row in structure:
            by_problem[row["problem"]].append(row)
        for problem, rows in by_problem.items():
            lines.append("")
            lines.append(f"  {problem.upper()}  ({len(rows)})")
            for row in rows:
                lines.append(f"      {row['chapter']} · {row['file']} · {row['sheet']}")
                lines.append(f"          {row['detail']}")

    lines.append("")
    path.write_text("\n".join(lines), encoding="utf-8")
    return path


## apply_review() - call this one yourself, after the review file is edited and saved

In [ ]:
"""
CELL: apply_review() - read the (possibly edited) review file back, and write
the confirmed decisions to translation dict.xlsx and value corrections.xlsx.

Nothing outside this cell writes to either file - running the checks and
writing the review is entirely read-only, on purpose, so the only path from
"found something" to "changed something" runs through a person having looked
at the file first.
"""

ID_LINE = re.compile(
    r"^\[(?P<letter>[LV])(?P<number>\d+)"
    r"(?: scope=(?P<scope>\w+))?"
    r" chapter=(?P<chapter>\S+)"
    r"(?: col_ar=(?P<col_ar>.+))?\]$"
)


def parse_review(path=None):
    """Every finding in the review file, with whatever is currently on its
    CORRECTION line - the suggestion, a person's edit, or SKIP.

    A line is read by *label*, not position, so reordering or wrapping the
    surrounding explanatory text never confuses it: only a line starting with
    RAW: or CORRECTION (once the ID line above it has been seen) is read.
    """
    path = path or REVIEW_PATH
    if not path.exists():
        raise FileNotFoundError(f"{path} does not exist - run this notebook first")

    findings = []
    current = None
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        match = ID_LINE.match(raw_line.strip())
        if match:
            current = {
                "kind": "label" if match["letter"] == "L" else "value",
                "id": f"{match['letter']}{match['number']}",
                "scope": match["scope"], "chapter": match["chapter"],
                "col_ar": match["col_ar"], "raw": None, "correction": None,
            }
            findings.append(current)
            continue
        if current is None:
            continue
        stripped = raw_line.strip()
        if stripped.startswith("RAW:"):
            current["raw"] = stripped[len("RAW:"):].strip()
        elif stripped.startswith("CORRECTION"):
            after_colon = stripped.split(":", 1)
            current["correction"] = after_colon[1].strip() if len(after_colon) > 1 else ""

    return findings


def dictionary_key(col_ar, val_ar):
    """The (column, value) pair that makes a dictionary row unique - with NaN
    normalized to None first. A float NaN never equals another float NaN
    (nan != nan is True), so leaving it as-is would make a column-defining
    row (val_ar genuinely blank, by design - see below) look "new" every
    time, defeating the whole point of checking for it: called twice, it
    would add itself twice.
    """
    return (col_ar, val_ar if pd.notna(val_ar) else None)


def update_dictionary(filled, backup=True):
    """Appends reviewed translations to translation dict.xlsx. Identical to
    notebook 3's function of the same name except for one fix that belongs in
    both: a row that only teaches a COLUMN name (val_ar and val_en both
    genuinely blank - a "column not in the dictionary" fix has no value to
    pair it with) used to be silently dropped by a blanket dropna() that
    assumed every row was a value translation. Kept as its own copy so this
    notebook needs nothing from notebook 3 either.
    """
    if not isinstance(filled, pd.DataFrame):
        filled = pd.read_excel(filled, engine="openpyxl")

    needed = ["col_ar", "val_ar", "col_en", "val_en"]
    missing = [c for c in needed if c not in filled.columns]
    if missing:
        raise ValueError(f"missing column(s) {missing}; expected {needed}")

    candidates = filled[needed].copy()
    is_column_row = candidates["val_ar"].isna() & candidates["val_en"].isna()
    column_rows = candidates[is_column_row
                             & candidates["col_ar"].notna() & candidates["col_en"].notna()]

    value_rows = candidates[~is_column_row].dropna()
    value_rows = value_rows[(value_rows["val_ar"].astype(str).str.strip() != "")
                            & (value_rows["val_en"].astype(str).str.strip() != "")]

    new_rows = pd.concat([column_rows, value_rows], ignore_index=True)
    if new_rows.empty:
        logger.warning("No completed rows to add.")
        return None

    dictionary = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")
    already_there = {dictionary_key(c, v)
                     for c, v in zip(dictionary["col_ar"], dictionary["val_ar"])}
    to_add = new_rows[~new_rows.apply(
        lambda r: dictionary_key(r["col_ar"], r["val_ar"]) in already_there, axis=1)]
    if to_add.empty:
        logger.info("Every row is already in the dictionary - nothing to add.")
        return dictionary

    if backup:
        stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        backup_path = TRANSLATION_DICT_PATH.with_name(
            f"{TRANSLATION_DICT_PATH.stem} backup {stamp}.xlsx")
        dictionary.to_excel(backup_path, index=False, engine="openpyxl")
        logger.info(f"Backed up the dictionary to {backup_path.name}")

    to_add = to_add.reindex(columns=dictionary.columns)
    if "status" in to_add.columns:
        to_add["status"] = "updated"

    updated = pd.concat([dictionary, to_add], ignore_index=True)
    updated.to_excel(TRANSLATION_DICT_PATH, index=False, engine="openpyxl")
    logger.info(f"Added {len(to_add):,} row(s) to {TRANSLATION_DICT_PATH.name} "
                f"({len(dictionary):,} -> {len(updated):,}).")
    return updated


def update_value_corrections(filled, backup=True):
    """Appends confirmed value readings to value corrections.xlsx - the same
    idempotent, back-up-first pattern as update_dictionary() above, keyed on
    (chapter, raw_value) instead of (col_ar, val_ar).
    """
    needed = ["chapter", "raw_value", "corrected_value"]
    missing = [c for c in needed if c not in filled.columns]
    if missing:
        raise ValueError(f"missing column(s) {missing}; expected {needed}")

    new_rows = filled[needed].dropna().copy()
    new_rows = new_rows[new_rows["corrected_value"].astype(str).str.strip() != ""]
    if new_rows.empty:
        logger.warning("No completed value corrections to add.")
        return None

    if VALUE_CORRECTIONS_PATH.exists():
        existing = pd.read_excel(VALUE_CORRECTIONS_PATH, engine="openpyxl")
    else:
        existing = pd.DataFrame(columns=["chapter", "raw_value", "corrected_value", "status", "date"])

    already_there = set(zip(existing.get("chapter", []), existing.get("raw_value", [])))
    to_add = new_rows[~new_rows.apply(
        lambda r: (r["chapter"], r["raw_value"]) in already_there, axis=1)]
    if to_add.empty:
        logger.info("Every value correction is already on file - nothing to add.")
        return existing

    if backup and VALUE_CORRECTIONS_PATH.exists():
        stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        backup_path = VALUE_CORRECTIONS_PATH.with_name(
            f"{VALUE_CORRECTIONS_PATH.stem} backup {stamp}.xlsx")
        existing.to_excel(backup_path, index=False, engine="openpyxl")
        logger.info(f"Backed up value corrections to {backup_path.name}")

    to_add["status"] = "updated"
    to_add["date"] = pd.Timestamp.now().strftime("%Y-%m-%d")
    updated = pd.concat([existing, to_add], ignore_index=True)
    updated.to_excel(VALUE_CORRECTIONS_PATH, index=False, engine="openpyxl")
    logger.info(f"Added {len(to_add):,} row(s) to {VALUE_CORRECTIONS_PATH.name} "
                f"({len(existing):,} -> {len(updated):,}).")
    return updated


def apply_review(path=None):
    """The whole review-and-apply loop's second half: read a saved,
    person-edited review file and act on every decision in it.
    """
    findings = parse_review(path)

    dictionary_rows = []
    value_rows = []
    skipped, blank, bad = [], [], []

    _, english_values = ENGLISH_VOCABULARY
    valid_columns = set(known_english_columns())
    column_map, _ = DICTIONARY_AR_TO_EN

    for finding in findings:
        correction = finding["correction"]
        if finding["raw"] is None:
            bad.append(f"{finding['id']}: no RAW line found - skipped")
            continue
        if correction is None or correction.strip() == "":
            blank.append(finding["id"])
            continue
        if correction.strip().upper() == SKIP:
            skipped.append(finding["id"])
            continue

        if finding["kind"] == "value":
            value_rows.append({"chapter": finding["chapter"], "raw_value": finding["raw"],
                               "corrected_value": correction})
            continue

        if finding["scope"] == "column":
            if correction not in valid_columns:
                bad.append(f"{finding['id']}: {correction!r} is not a known English column name - skipped")
                continue
            dictionary_rows.append({"col_ar": finding["raw"], "val_ar": None,
                                    "col_en": correction, "val_en": None})
        else:  # value or source
            col_ar = finding["col_ar"]
            col_en = column_map.get(col_ar, col_ar)
            dictionary_rows.append({"col_ar": col_ar, "val_ar": finding["raw"],
                                    "col_en": col_en, "val_en": correction})

    print(f"Findings in the review file: {len(findings)}")
    print(f"  to apply:  {len(dictionary_rows) + len(value_rows)}")
    print(f"  skipped:   {len(skipped)}")
    print(f"  left blank (treated as skipped): {len(blank)}")
    if bad:
        print(f"  could not use: {len(bad)}")
        for message in bad:
            print(f"    {message}")

    if dictionary_rows:
        update_dictionary(pd.DataFrame(dictionary_rows))
    if value_rows:
        update_value_corrections(pd.DataFrame(value_rows))
    if not dictionary_rows and not value_rows:
        logger.info("Nothing to apply.")

    return {"applied": len(dictionary_rows) + len(value_rows), "skipped": len(skipped),
           "blank": len(blank), "bad": len(bad)}


## Run

In [ ]:
"""
CELL: Main run - read every questionnaire, write the review file.
"""
run_checks()

labels = aggregate_labels()
values = aggregate_values()

path = write_review(labels, values, STRUCTURE_PROBLEMS)

print("\n" + "=" * 70)
print("REVIEW FILE WRITTEN")
print("=" * 70)
print(f"  {path}")
print(f"\n  {len(labels):>5,}  label(s) not found in the dictionary, exactly")
print(f"  {len(values):>5,}  value(s) that are not plain numbers")
print(f"  {len(STRUCTURE_PROBLEMS):>5,}  structural problem(s) - reporting only")
print(f"\nOpen it, edit or accept each CORRECTION line, save, then tell Claude:")
print(f"  apply {path.name}")
